# Bài 2 (Trang 37, 38) - Online Retail (United Kingdom)
## Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
%matplotlib inline

df = pd.read_excel("../Dataset/Online Retail.xlsx")

### Các bước xử lý của câu 1 → 9 (chuẩn bị dữ liệu)

In [3]:
df["Description"] = df["Description"].str.strip()
df = df.dropna(axis=0, subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df[~df["InvoiceNo"].str.contains("C")]

grouped = (df[df["Country"] == "United Kingdom"]
           .groupby(["InvoiceNo", "Description"])["Quantity"]
           .sum())

basket = grouped.unstack().reset_index().fillna(0).set_index("InvoiceNo")

def encode_units(x):
    return 1 if x >= 1 else 0

# Tương đương basket.map(encode_units) nhưng vector hóa cho nhanh; kiểu bool theo khuyến nghị của mlxtend
basket_sets = basket >= 1

basket_sets = basket_sets.drop(columns=["POSTAGE"], errors="ignore")

start = time.time()
frequent_itemsets = apriori(basket_sets, min_support=0.03, use_colnames=True)
apriori_time = time.time() - start

### Câu 10: Tạo luật kết hợp với min_conf = 50% và in ra các luật

In [4]:
rules = association_rules(frequent_itemsets, num_itemsets=len(basket_sets),
                          metric="confidence", min_threshold=0.5)

In [5]:
rules.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   antecedents         10 non-null     object 
 1   consequents         10 non-null     object 
 2   antecedent support  10 non-null     float64
 3   consequent support  10 non-null     float64
 4   support             10 non-null     float64
 5   confidence          10 non-null     float64
 6   lift                10 non-null     float64
 7   representativity    10 non-null     float64
 8   leverage            10 non-null     float64
 9   conviction          10 non-null     float64
 10  zhangs_metric       10 non-null     float64
 11  jaccard             10 non-null     float64
 12  certainty           10 non-null     float64
 13  kulczynski          10 non-null     float64
dtypes: float64(12), object(2)
memory usage: 1.2+ KB


In [6]:
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

In [7]:
print("Số luật tìm được:", len(rules))
for i, r in rules.iterrows():
    print(f"Rule {i + 1}: {r['antecedents']} => {r['consequents']} "
          f"(support = {r['support']:.3f}, confidence = {r['confidence']:.3f}, lift = {r['lift']:.3f})")

Số luật tìm được: 10
Rule 1: ALARM CLOCK BAKELIKE GREEN => ALARM CLOCK BAKELIKE RED (support = 0.030, confidence = 0.643, lift = 12.900)
Rule 2: ALARM CLOCK BAKELIKE RED => ALARM CLOCK BAKELIKE GREEN (support = 0.030, confidence = 0.605, lift = 12.900)
Rule 3: GREEN REGENCY TEACUP AND SAUCER => PINK REGENCY TEACUP AND SAUCER (support = 0.031, confidence = 0.618, lift = 16.404)
Rule 4: PINK REGENCY TEACUP AND SAUCER => GREEN REGENCY TEACUP AND SAUCER (support = 0.031, confidence = 0.821, lift = 16.404)
Rule 5: GREEN REGENCY TEACUP AND SAUCER => ROSES REGENCY TEACUP AND SAUCER (support = 0.038, confidence = 0.751, lift = 14.640)
Rule 6: ROSES REGENCY TEACUP AND SAUCER => GREEN REGENCY TEACUP AND SAUCER (support = 0.038, confidence = 0.732, lift = 14.640)
Rule 7: JUMBO  BAG BAROQUE BLACK WHITE => JUMBO BAG RED RETROSPOT (support = 0.031, confidence = 0.626, lift = 6.033)
Rule 8: JUMBO BAG PINK POLKADOT => JUMBO BAG RED RETROSPOT (support = 0.042, confidence = 0.677, lift = 6.524)
Rule 9: 

### Nhận xét
- Với min_conf = 50% thu được 10 luật, ví dụ: GREEN REGENCY TEACUP AND SAUCER => ROSES REGENCY TEACUP AND SAUCER (conf ≈ 0.75), PINK REGENCY TEACUP AND SAUCER => GREEN REGENCY TEACUP AND SAUCER (conf ≈ 0.82).
- Các luật cho thấy khách hàng thường mua các sản phẩm cùng bộ/cùng mẫu với màu khác nhau.